# CalibrateQwen 01: hard-label or teacher-completion SFT
We run one structured-output SFT variant and retain the Tinker checkpoint log for evaluation.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass

In [ ]:
METHOD = 'teacher'  # 'hard_label' or 'teacher'
CONFIDENCE_FORMAT = 'numeric'  # 'numeric', 'bucket', or 'implicit'
MAX_STEPS = None  # Set 3 for a paid pipeline smoke test
MODEL = 'Qwen/Qwen3.5-4B'
RUN_NAME = f'sft_{METHOD}_{CONFIDENCE_FORMAT}'
RUN_ROOT = Path('/content/calibrate_qwen_runs')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
from training.prepare_training_data import prepare_training_file
from training.train_sft import SFTConfig, train_sft

conversation_file = f'artifacts/training/{METHOD}_{CONFIDENCE_FORMAT}.jsonl'
prepare_training_file(
    output_path=conversation_file,
    variant=METHOD,
    confidence_format=CONFIDENCE_FORMAT,
    abstention_threshold=0.6,
)
config = SFTConfig(
    conversation_file=conversation_file,
    log_path=str(RUN_ROOT / RUN_NAME),
    model_name=MODEL,
    batch_size=32,
    learning_rate=2e-4,
    lora_rank=32,
    max_steps=MAX_STEPS,
    wandb_project='calibrate-qwen' if os.environ.get('WANDB_API_KEY') else None,
    wandb_name=RUN_NAME,
)
config

In [ ]:
await train_sft(config)

In [ ]:
checkpoint_log = Path(config.log_path) / 'checkpoints.jsonl'
print(checkpoint_log.read_text() if checkpoint_log.exists() else 'Checkpoint log will appear after the first save.')